# QPIE encoding and two-dimensional quantum Fourier transform

This local edition uses Qiskit 2.x. Exact statevector decoding is the default, so the examples do not need millions of measurement shots or an IBM Quantum account.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import QPIE
import FRQI

QPIE stores normalized pixel values in quantum-state amplitudes. An $N\times N$ image uses $2\log_2(N)$ qubits. The synthetic image below keeps this notebook self-contained.

In [ ]:
image = np.zeros((32, 32), dtype=float)
image[5:8, 6:9] = 1.0
image[23:27, 24:28] = 0.65

plt.imshow(image, cmap="gray")
plt.title("Input image")
plt.axis("off");

In [ ]:
qpie = QPIE.qpie_circuit(image)
decoded = QPIE.decode_out(qpie, np.linalg.norm(image))

print(f"Qubits: {qpie.num_qubits}")
print(f"Exact QPIE MSE: {QPIE.mse(image, decoded):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Input")
axes[1].imshow(decoded, cmap="gray")
axes[1].set_title("Exact local decode")
for axis in axes:
    axis.axis("off")
plt.tight_layout();

The two-dimensional QFT is separable: one QFT gate is applied to the row register and another to the column register. QPIE's decoder restores the classical FFT magnitude scale.

In [ ]:
qft_circuit = QPIE.apply_qft_2d(qpie)
quantum_fft = QPIE.decode_out(
    qft_circuit,
    np.linalg.norm(image),
    fourier=True,
)
classical_fft = np.abs(np.fft.fft2(image))

print(f"QFT-vs-FFT MSE: {QPIE.mse(quantum_fft, classical_fft):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(np.log1p(quantum_fft), cmap="gray")
axes[0].set_title("QPIE + QFT")
axes[1].imshow(np.log1p(classical_fft), cmap="gray")
axes[1].set_title("NumPy FFT")
for axis in axes:
    axis.axis("off")
plt.tight_layout();

FRQI uses a color qubit and position qubits. The optimized implementation uses one uniformly controlled Y-rotation instead of a separate multi-controlled rotation and barrier for every pixel.

In [ ]:
frqi = FRQI.encode_image(image)
frqi_decoded = FRQI.decode_out(frqi)

print(f"FRQI qubits: {frqi.num_qubits}")
print(f"Exact FRQI MSE: {FRQI.mse(image, frqi_decoded):.3e}")

## Optional sampling experiment

Exact decoding is best for fast local development. Use `shots=` only when measurement noise is part of the experiment.

In [ ]:
small_image = np.arange(16, dtype=float).reshape(4, 4) / 15
small_qpie = QPIE.qpie_circuit(small_image)
sampled = QPIE.decode_out(
    small_qpie,
    np.linalg.norm(small_image),
    shots=16_384,
    seed=12345,
)

print(f"Sampled MSE (16,384 shots): {QPIE.mse(small_image, sampled):.3e}")

## Scaling note

The qubit count grows logarithmically with image width, but arbitrary state preparation and classical statevector memory still scale with the number of pixels. Exact local simulation avoids shot noise and is normally much faster for research and debugging.